<a href="https://colab.research.google.com/github/07-khanh/Hands-On-ML/blob/main/spam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tarfile
from pathlib import Path
import urllib.request
from google.colab import drive

def fetch_spam_data():
  drive.mount('/content/drive')

  spam_root = "https://spamassassin.apache.org/old/publiccorpus/"
  ham_url = spam_root + "20030228_easy_ham.tar.bz2"
  spam_url = spam_root + "20030228_spam.tar.bz2"

  spam_path = Path("/content/drive/MyDrive/KaggleData/spam")
  spam_path.mkdir(parents=True, exist_ok=True)



  for dir_name, tar_name, url in (("easy_ham", "ham", ham_url),
                                  ("spam", "spam", spam_url)):
    if not (spam_path / dir_name).is_dir():
      path = (spam_path / tar_name).with_suffix(".tar.bz2")
      print("Download", path)
      urllib.request.urlretrieve(url, path)
      with tarfile.open(path) as tar_bz2_file:
        tar_bz2_file.extractall(path=spam_path)

      print("Deleting archive:", path)
      path.unlink()

  return [spam_path / dir_name for dir_name in ("easy_ham", "spam")]

In [3]:
ham_dir, spam_dir = fetch_spam_data()

Mounted at /content/drive


In [4]:
ham_filenames = [f for f in ham_dir.iterdir() if len(f.name) > 20]
spam_filenames = [f for f in spam_dir.iterdir() if len(f.name) > 20]

In [5]:
len(ham_filenames), len(spam_filenames)

(2500, 500)

In [6]:
import email
import email.policy
from email.parser import BytesParser

def load_email(filepath):
  with open(filepath, 'rb') as f:
    return BytesParser(policy=email.policy.default).parse(f)

In [7]:
ham_emails = [load_email(filepath) for filepath in ham_filenames]
spam_emails = [load_email(filepath) for filepath in spam_filenames]

In [8]:
print(ham_emails[1].get_content().strip())

I have searched the list but did not find any info on this. 

How do I setup multiple spamd machines so that spamc load balances - or 
anything similar? 

Duane.


-------------------------------------------------------
This sf.net email is sponsored by:ThinkGeek
Welcome to geek heaven.
http://thinkgeek.com/sf
_______________________________________________
Spamassassin-devel mailing list
Spamassassin-devel@lists.sourceforge.net
https://lists.sourceforge.net/lists/listinfo/spamassassin-devel


In [9]:
def get_email_structure(email):
  if isinstance(email, str):
    return email
  payload = email.get_payload()
  if isinstance(payload, list):
    multipart = ', '.join([get_email_structure(sub_email)
                            for sub_email in payload])
    return f"multipart({multipart})"
  else:
    return email.get_content_type()

In [10]:
from collections import Counter

def structures_counter(emails):
  structures = Counter()
  for email in emails:
    structure = get_email_structure(email)
    structures[structure] += 1
  return structures

In [11]:
structures_counter(ham_emails).most_common()

[('text/plain', 2408),
 ('multipart(text/plain, application/pgp-signature)', 66),
 ('multipart(text/plain, text/html)', 8),
 ('multipart(text/plain, text/plain)', 4),
 ('multipart(text/plain)', 3),
 ('multipart(text/plain, application/octet-stream)', 2),
 ('multipart(text/plain, multipart(text/plain))', 1),
 ('multipart(text/plain, application/x-pkcs7-signature)', 1),
 ('multipart(text/plain, multipart(text/plain, text/plain), text/rfc822-headers)',
  1),
 ('multipart(multipart(text/plain, text/plain, text/plain), application/pgp-signature)',
  1),
 ('multipart(text/plain, video/mng)', 1),
 ('multipart(text/plain, application/x-java-applet)', 1),
 ('multipart(text/plain, multipart(text/plain, text/plain), multipart(multipart(text/plain, application/x-pkcs7-signature)))',
  1),
 ('multipart(text/plain, text/enriched)', 1),
 ('multipart(text/plain, application/ms-tnef, text/plain)', 1)]

In [12]:
structures_counter(spam_emails).most_common()

[('text/plain', 218),
 ('text/html', 183),
 ('multipart(text/plain, text/html)', 45),
 ('multipart(text/html)', 20),
 ('multipart(text/plain)', 19),
 ('multipart(multipart(text/html))', 5),
 ('multipart(text/plain, image/jpeg)', 3),
 ('multipart(text/html, application/octet-stream)', 2),
 ('multipart(multipart(text/html), application/octet-stream, image/jpeg)', 1),
 ('multipart(multipart(text/plain, text/html), image/gif)', 1),
 ('multipart(text/plain, application/octet-stream)', 1),
 ('multipart(text/html, text/plain)', 1),
 ('multipart/alternative', 1)]

In [13]:
for header, value in spam_emails[0].items():
  print(header, ":", value)

Return-Path : <wsup@playful.com>
Delivered-To : zzzz@localhost.spamassassin.taint.org
Received : from localhost (localhost [127.0.0.1])	by phobos.labs.spamassassin.taint.org (Postfix) with ESMTP id B8E8D43F99	for <zzzz@localhost>; Thu, 22 Aug 2002 11:16:59 -0400 (EDT)
Received : from mail.webnote.net [193.120.211.219]	by localhost with POP3 (fetchmail-5.9.0)	for zzzz@localhost (single-drop); Thu, 22 Aug 2002 16:16:59 +0100 (IST)
Received : from smtp.easydns.com (smtp.easydns.com [205.210.42.30])	by webnote.net (8.9.3/8.9.3) with ESMTP id QAA05397	for <zzzz@spamassassin.taint.org>; Thu, 22 Aug 2002 16:13:20 +0100
Received : from 200.161.16.132 (unknown [210.19.113.130])	by smtp.easydns.com (Postfix) with SMTP id 694632EE5A	for <zzzz@spamassassin.taint.org>; Thu, 22 Aug 2002 11:12:57 -0400 (EDT)
Received : from unknown (52.127.142.42) by rly-xl04.mx.aol.com with smtp; Aug, 22 2002 8:02:13 AM +0400
Received : from [176.244.234.14] by smtp-server6.tampabay.rr.com with local; Aug, 22 2002 6

In [14]:
spam_emails[0]['Subject']

'Re: Fw: User Name & Password to Membership To 5 Sites zzzz@spamassassin.taint.org pviqg'

In [16]:
import numpy as np
from sklearn.model_selection import train_test_split

X = np.array(ham_emails + spam_emails, dtype=object)
y = np.array([0] * len(ham_emails) + [1] * len(spam_emails))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)

In [17]:
X_train

array([<email.message.EmailMessage object at 0x7fbb2647acf0>,
       <email.message.EmailMessage object at 0x7fbb2713cce0>, ...,
      dtype=object)

In [44]:
import re
from html import unescape

def html_to_plain_text(html):
    text = re.sub('<head.*?>.*?</head>', '', html, flags=re.M | re.S | re.I)
    text = re.sub('<a\s.*?>', ' HYPERLINK ', text, flags=re.M | re.S | re.I)
    text = re.sub('<.*?>', '', text, flags=re.M | re.S)
    text = re.sub(r'(\s*\n)+', '\n', text, flags=re.M | re.S)
    return unescape(text)

<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1327/3203418293.py:6: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('<a\s.*?>', ' HYPERLINK ', text, flags=re.M | re.S | re.I)


In [32]:
html_spam_emails = [email for email in X_train[y_train == 1]
                    if get_email_structure(email) == 'text/html']
sample_html_spam = html_spam_emails[6]
print(sample_html_spam.get_content().strip()[:1000], "...")

<HTML>
<FONT COLOR ="#000000"> <STRONG><H3>Hello! : </H3></FONT>
<p>

<p>
<FONT COLOR ="FF0000"><H1> "Dmm": Discount Mortgage Millionaire<BR> 
Program: <BR></H1></FONT>
<p>
<FONT COLOR ="#008000"><H1>A Special Invitation To Wealth !!! </FONT></H1>
<P>	
<FONT COLOR ="#0000FF"><H2><FONT TYPE="ARIEL"><STRONG>Help Us Contact Home Owners In Your City And Pocket <BR>
"$1000" / Deal, Easy Cash!!! </FONT></H2></STRONG>

<p>
<FONT COLOR ="#000000"><H5>
</H5></FONT>
<p>
<p>
<FONT COLOR ="#000000"><H5> To:
<P>
This e-mail is addressed to  all people who have tried different internet <BR>
programs and failed,  who are tired of working hard, but getting nowhere, <BR>
who are unemployed,  and all those who have jobs but need extra income, <BR>
and all those who are sick and tired of being sick and tired. </H5></FONT>

<p>
<FONT COLOR ="FF0000"><H2> How It Works: </H2></FO ...


In [33]:
print(html_to_plain_text(sample_html_spam.get_content()).strip()[:1000], "...")

Hello! :
 "Dmm": Discount Mortgage Millionaire
Program:
A Special Invitation To Wealth !!!
Help Us Contact Home Owners In Your City And Pocket
"$1000" / Deal, Easy Cash!!!
 To:
This e-mail is addressed to  all people who have tried different internet
programs and failed,  who are tired of working hard, but getting nowhere,
who are unemployed,  and all those who have jobs but need extra income,
and all those who are sick and tired of being sick and tired.
 How It Works:
 We buy Real Estate Notes and Trust Deeds. We also represent investors
who invest in them.
When you apply, you can help us contact Home Owners in your city and
on the internet.
All you do is call or e-mail them or send a simple but Powerful letter and form to them to complete and return back to us.
If they have Real Estate Notes or Trust Deeds, we'll either buy them or refer
them to our associ ...


In [45]:
def email_to_text(email):
  html = None
  for part in email.walk():
    ctype = part.get_content_type()
    # not taking types other than plain text and html
    if not ctype in ('text/plain', 'text/html'):
      continue

    try:
      content = part.get_content()
    except: # in case of encoding issues
      content = str(part.get_payload())

    # prioritize plain text content
    if ctype == 'text/plain':
      return content
    else:
      html = content

  if html:
    return html_to_plain_text(html)

In [49]:
print(email_to_text(spam_emails[1])[:1000], "...")


          --- The most comprehensive adult match making service
                Check some of our  actual pictures from real members!
                                    Welcome to one of the Internet's premier adult match making services where people just like yourself can view and place personal advertisements which are viewed by thousands daily!
                Web Adult Classifieds has thousands of ads - something for everyone, male or felmale!
                   
                   HYPERLINK Click here to be convinced  
                  (will open in a new window for your convenience)
      ©XxxMatch.net 2002
4199lrjl7 ...


In [50]:
import nltk

stemmer = nltk.PorterStemmer()
for word in ("Computations", "Computation", "Computing", "Computed", "Compute",
             "Compulsive"):
  print(word, '=>', stemmer.stem(word))

Computations => comput
Computation => comput
Computing => comput
Computed => comput
Compute => comput
Compulsive => compuls


In [52]:
import sys

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules


# if running this notebook on Colab or Kaggle, we just pip install urlextract
if IS_COLAB or IS_KAGGLE:
  %pip install -q -U urlextract

In [53]:
import urlextract

url_extractor = urlextract.URLExtract()
some_text = "Will it detect github.com and https://youtu.be/7Pq-S557XQU?t=3m32s"
print(url_extractor.find_urls(some_text))

['github.com', 'https://youtu.be/7Pq-S557XQU?t=3m32s']


In [54]:
from sklearn.base import BaseEstimator, TransformerMixin

class EmailToWordCounterTransformer(BaseEstimator, TransformerMixin):
  def __init__(self, strip_header=True, lower_case=True,
               remove_punctuation=True, replace_urls=True,
               replace_numbers=True, stemming=True):
    self.strip_header = strip_header
    self.lower_case = lower_case
    self.remove_punctuation = remove_punctuation
    self.replace_urls = replace_urls
    self.replace_numbers = replace_numbers
    self.stemming = stemming
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    X_transformed = []
    for email in X:
      text = email_to_text(email) or ""
      if self.lower_case:
        text = text.lower()
      if self.replace_urls and url_extractor is not None:
        urls = list(url_extractor.find_urls(text))
        urls.sort(key=lambda url: len(url), reverse=True)
        for url in urls:
          text = text.replace(url, " URL ")
      if self.replace_numbers:
        text = re.sub(r'\d+(?:\.\d*)?(?:[eE][+-]?\d+)?', 'NUMBER', text)
      if self.remove_punctuation:
        text = re.sub(r'\W+', ' ', text, flags=re.M)
      word_counts = Counter(text.split())
      if self.stemming and stemmer is not None:
        stemmed_word_counts = Counter()
        for word, count in word_counts.items():
          stemmed_word = stemmer.stem(word)
          stemmed_word_counts[stemmed_word] += count
        word_counts = stemmed_word_counts
      X_transformed.append(word_counts)
    return np.array(X_transformed)

In [56]:
X_few = X_train[:3]
X_few_wordcounts = EmailToWordCounterTransformer().fit_transform(X_few)
X_few_wordcounts[0]

Counter({'url': 9,
         'date': 1,
         'number': 29,
         'numbertnumb': 1,
         'one': 2,
         'year': 2,
         'ago': 2,
         'today': 3,
         'my': 7,
         'now': 1,
         'former': 1,
         'manag': 1,
         'told': 1,
         'me': 2,
         'to': 7,
         'shut': 1,
         'down': 1,
         'weblog': 1,
         'and': 4,
         'remov': 1,
         'all': 1,
         'trace': 1,
         'of': 2,
         'it': 1,
         'from': 1,
         'server': 1,
         'he': 1,
         'tri': 1,
         'convinc': 1,
         'that': 1,
         'the': 4,
         'internet': 1,
         'wa': 1,
         'too': 1,
         'small': 1,
         'mix': 1,
         'profession': 1,
         'person': 1,
         'i': 2,
         'gave': 1,
         'him': 1,
         'answer': 1,
         'rest': 1,
         'is': 1,
         'histori': 1,
         'celebr': 1,
         'thi': 4,
         'anniversari': 1,
         'would': 1,


In [61]:
from scipy.sparse import csr_matrix

class WordCounterToVectorTransformer(BaseEstimator, TransformerMixin):
  def __init__(self, vocabulary_size=1000):
    self.vocabulary_size = vocabulary_size
  def fit(self, X, y=None):
    total_count = Counter()
    for word_count in X:
      for word, count in word_count.items():
        total_count[word] += min(count, 10)
    most_common = total_count.most_common()[:self.vocabulary_size]
    self.vocabulary_ = {word: index + 1
                        for index, (word, count) in enumerate(most_common)}
    return self
  def transform(self, X, y=None):
    rows = []
    cols = []
    data = []
    for row, word_count in enumerate(X):
      for word, count in word_count.items():
        rows.append(row)
        cols.append(self.vocabulary_.get(word, 0))
        data.append(count)
    return csr_matrix((data, (rows, cols)),
                      shape=(len(X), self.vocabulary_size+1))

In [60]:
vocab_transformer = WordCounterToVectorTransformer(vocabulary_size=10)
X_few_vectors = vocab_transformer.fit_transform(X_few_wordcounts)
X_few_vectors.toarray()

array([[ 87,  29,   9,   7,   4,   2,   4,   7,   0,   2,   1],
       [ 20,   0,   2,   1,   1,   0,   0,   0,   0,   1,   1],
       [164,   8,   6,   2,   5,   8,   6,   1,   8,   3,   4]])

In [62]:
vocab_transformer.vocabulary_

{'number': 1,
 'url': 2,
 'to': 3,
 'the': 4,
 'i': 5,
 'thi': 6,
 'my': 7,
 'razor': 8,
 'of': 9,
 'is': 10}

In [63]:
from sklearn.pipeline import Pipeline

preprocessing = Pipeline([
    ('email_to_wordcount', EmailToWordCounterTransformer()),
    ('wordcount_to_vector', WordCounterToVectorTransformer()),
])
X_train_transformed = preprocessing.fit_transform(X_train)

In [66]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

log_clf = LogisticRegression(max_iter=1000, random_state=42)
score = cross_val_score(log_clf, X_train_transformed, y_train, cv=3)

In [67]:
score

array([0.9875 , 0.99125, 0.98375])

In [69]:
from sklearn.metrics import precision_score, recall_score

X_test_transformed = preprocessing.transform(X_test)

log_clf = LogisticRegression(max_iter=1000, random_state=42)
log_clf.fit(X_train_transformed, y_train)

y_pred = log_clf.predict(X_test_transformed)

print(f"Precision: {precision_score(y_test, y_pred):.2%}")
print(f"Recall: {recall_score(y_test, y_pred):.2%}")

Precision: 100.00%
Recall: 96.84%


In [70]:
log_clf.score(X_test_transformed, y_test)

0.995